# Tutorial: Syntactic Parsing with spaCy
## NLP Course - Lab 4: Constituency & Dependency Analysis

### Introduction
This tutorial is designed to explore the structural analysis of natural language. We will cover:
1. **Dependency Parsing**: Focusing on the relationships between individual words.
2. **Constituency Parsing**: Focusing on the hierarchical phrase structure.

### Prerequisites
You will need `spacy` and `benepar` (Berkeley Neural Parser).

### 1. Setup and Environment
We will use a transformer-optimized model for spaCy and integrate `benepar` for constituency parsing, as spaCy does not provide this out-of-the-box.

In [ ]:
!pip install spacy benepar
import spacy
from spacy import displacy
import benepar

# Download models
spacy.cli.download('en_core_web_md')
benepar.download('benepar_en3')

# Load the spaCy model and add the benepar component to the pipeline
nlp = spacy.load('en_core_web_md')
if 'benepar' not in nlp.pipe_names:
    nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

print('\nPipeline components:', nlp.pipe_names)

### 2. Dependency Parsing
Dependency parsing identifies the **head** words and their **dependents**. In Computer Science terms, this results in a **Directed Acyclic Graph (DAG)**.

#### 2.1 Visualization
We use `displacy` to view the grammatical dependencies.

In [ ]:
doc = nlp('The experienced computer science students are analyzing complex linguistic structures.')

# Visualize the dependency tree
displacy.render(doc, style='dep', jupyter=True, options={'distance': 120})

#### 2.2 Navigating the Dependency Graph
Every `Token` in a spaCy `Doc` has attributes that allow you to traverse the tree: `.head`, `.children`, `.lefts`, and `.rights`.

In [ ]:
print(f"{'Text':<12} | {'Dep':<10} | {'Head':<12} | {'Children'}")
print('-' * 60)

for token in doc:
    children = [child.text for child in token.children]
    print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12} | {children}")

### 3. Constituency (Phrase Structure) Parsing
Constituency parsing breaks sentences into sub-phrases (constituents). This follows Context-Free Grammar (CFG) rules.

#### 3.1 Inspecting the Tree Structure
With `benepar`, we can access labels like `S` (Sentence), `NP` (Noun Phrase), and `VP` (Verb Phrase).

In [ ]:
sentence = 'The student solved the problem with a clever algorithm.'
doc = nlp(sentence)
sent = list(doc.sents)[0]

def print_tree(node, depth=0):
    # Extract labels from the benepar extension
    label = node._.labels[0] if node._.labels else 'TOKEN'
    print('  ' * depth + f'[{label}] {node.text}')
    for child in node._.children:
        print_tree(child, depth + 1)

print_tree(sent)

#### 3.2 Handling Ambiguity
Syntactic ambiguity is a major challenge in NLP. Observe how the parser handles Prepositional Phrase (PP) attachment.

In [ ]:
ambig_1 = nlp('He ate the salad with a fork.')
ambig_2 = nlp('He ate the salad with croutons.')

print('--- Sentence 1 Tree ---')
print_tree(list(ambig_1.sents)[0])
print('\n--- Sentence 2 Tree ---')
print_tree(list(ambig_2.sents)[0])

### 4. Comparison Summary

| Feature | Dependency | Constituency |
| :--- | :--- | :--- |
| **Focus** | Word-to-word relations | Hierarchical grouping |
| **Application** | Information Extraction | Grammar Checking |
| **Logic** | Head/Dependent | Part/Whole |